# `aidp-bds-hive` live test — Kerberos
**Live-test row 10.** Requires `kinit` on the cluster image (TBD; this notebook will fail fast with a clear error if not).


In [ ]:
import sys, os
# Adjust this if you've uploaded the plugin scripts/ dir elsewhere.
sys.path.insert(0, '/Workspace/Shared/oracle_ai_data_platform_connectors/scripts')


In [ ]:
from oracle_ai_data_platform_connectors.jdbc import build_hive_jdbc_url, spark_hive_jdbc_options
from oracle_ai_data_platform_connectors.jdbc.hive import kerberos_kinit

kerberos_kinit(
    principal=os.environ['BDS_KRB_PRINCIPAL'],
    keytab_path=os.environ['BDS_KRB_KEYTAB_PATH'],  # MUST be /tmp/...
)
url = build_hive_jdbc_url(
    host=os.environ['BDS_HS2_HOST'], port=10000,
    database=os.environ.get('BDS_HS2_DATABASE', 'default'),
    auth='kerberos',
    principal=f"hive/{os.environ['BDS_HS2_HOST']}@{os.environ['BDS_HIVE_REALM']}",
)
opts = spark_hive_jdbc_options(url=url)


In [ ]:
df = spark.read.format('jdbc').options(**opts).option('dbtable', os.environ['BDS_TABLE_FOR_TEST']).load()
df.show(5)


In [ ]:
# Live-test driver parses this final cell's stdout for the JSON summary.
import json, time
summary = {
    'connector': 'aidp-bds-hive',
    'auth': 'kerberos',
    'rows': int(df.count()),
    'schema': sorted([f.name for f in df.schema.fields]),
    'timestamp_utc': int(time.time()),
}
print('AIDP_LIVE_TEST_RESULT_BEGIN')
print(json.dumps(summary, indent=2))
print('AIDP_LIVE_TEST_RESULT_END')
